In [7]:
import pandas as pd
import json
import os
import sys
from dotenv import load_dotenv



In [8]:
# 1. Load your custom cleaning pipeline
sys.path.append(os.path.abspath('..'))
from src.features import process_raw_to_clean

# Load the .env file so Hugging Face can find the HF_TOKEN
from pathlib import Path
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

# NOW we can import the ML libraries safely
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [9]:
# 2. Load and clean the data
file_path = '../data/raw/20260213_technology_news.json'
with open(file_path, 'r') as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data['articles'])
df_clean = process_raw_to_clean(df_raw)

# BERTopic requires a simple Python list of strings
docs = df_clean['text_cleaned'].tolist()

print(f"Ready to cluster {len(docs)} articles...\n")



Ready to cluster 43 articles...



In [10]:
# 3. Load the Embedding Model
# "all-MiniLM-L6-v2" is highly optimised, fast, and won't crash my Mac's RAM
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1742.01it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# 4.
from sklearn.feature_extraction.text import CountVectorizer
# Define the Vectorizer to remove english stopwords and look for 1 or 2-word phrases
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

# Initialise and train BERTopic
# Note: Because the free NewsAPI tier only gives me ~100 articles a day,
# I'm setting a low min_topic_size - this will ensure the model can actually find groups
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=5
)

topics, probs = topic_model.fit_transform(docs)

In [14]:
# 5. View the Discovered Topics
topic_info = topic_model.get_topic_info()
display(topic_info.head(10))

,Topic,Count,Name,Representation,Representative_Docs
0,-1,15,-1_new_release_ai_maude,"[new, release, ai, maude, quietly drops, quarr...",[until dawn and the quarry developer announces...
1,0,12,0_apple_galaxy_coming_updates,"[apple, galaxy, coming, updates, ios, stock, s...",[galaxy s ultra release date samsung confirms ...
2,1,11,1_game_arc_play reanimal_play,"[game, arc, play reanimal, play, new riot, pla...",[the switch s gameshare multiplayer turns this...
3,2,5,2_update_developer_manufacture_sky remnant,"[update, developer, manufacture, sky remnant, ...",[update highguard developer wildlight entertai...
